In [2]:
import polars as pl
from pathlib import Path

# Anchor to repo root, so it works whether the kernel runs from
# the repo root or the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUT = PROJECT_ROOT / "data" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)


def load_master() -> pl.DataFrame:
    return pl.read_parquet(PROCESSED / "master_flight_dashboard.parquet")

In [3]:
def flights_by_route(df: pl.DataFrame) -> pl.DataFrame:
    """Top routes by number of flights."""
    return (
        df.lazy()
        .group_by("route_code", "route_origin", "route_destination")
        .agg(
            pl.len().alias("total_flights"),
            pl.col("route_distance").mean().alias("avg_distance_km"),
            pl.col("route_flight_minutes").mean().alias("avg_duration_min"),
        )
        .sort("total_flights", descending=True)
        .collect()
    )


def flights_by_continent(df: pl.DataFrame) -> pl.DataFrame:
    """Flight volume by origin continent."""
    return (
        df.lazy()
        .group_by("origin_continent")
        .agg(
            pl.len().alias("total_flights"),
            pl.col("route_distance").sum().alias("total_distance_km"),
        )
        .sort("total_flights", descending=True)
        .collect()
    )

In [4]:
def fleet_usage(df: pl.DataFrame) -> pl.DataFrame:
    """Flights and distance per aircraft."""
    return (
        df.lazy()
        .group_by("airplane", "airplane_model")
        .agg(
            pl.len().alias("total_flights"),
            pl.col("route_distance").sum().alias("total_distance_km"),
        )
        .sort("total_flights", descending=True)
        .collect()
    )


def fleet_maintenance(df: pl.DataFrame) -> pl.DataFrame:
    """Aircraft age vs maintenance hours for scatter plot."""
    return (
        df.lazy()
        .select("airplane", "airplane_model", "airplane_build_date",
                "airplane_maintenance_flight_hours", "airplane_total_flight_distance")
        .unique()
        .with_columns(
            (2026 - pl.col("airplane_build_date").cast(pl.Date).dt.year())
            .alias("aircraft_age_years")
        )
        .collect()
    )

In [5]:
df = load_master()

r1 = flights_by_route(df)
r1.write_parquet(OUTPUT / "flights_by_route.parquet")
print(f"flights_by_route: {r1.shape}")

r2 = flights_by_continent(df)
r2.write_parquet(OUTPUT / "flights_by_continent.parquet")
print(f"flights_by_continent: {r2.shape}")

r3 = fleet_usage(df)
r3.write_parquet(OUTPUT / "fleet_usage.parquet")
print(f"fleet_usage: {r3.shape}")

r4 = fleet_maintenance(df)
r4.write_parquet(OUTPUT / "fleet_maintenance.parquet")
print(f"fleet_maintenance: {r4.shape}")

print("All outputs saved to data/output/")

flights_by_route: (592, 6)
flights_by_continent: (3, 3)
fleet_usage: (272, 4)
fleet_maintenance: (272, 6)
All outputs saved to data/output/


In [ ]:

for name in ["flights_by_route", "flights_by_continent", "fleet_usage", "fleet_maintenance"]:
    result = pl.read_parquet(OUTPUT / f"{name}.parquet")
    print(f"{name}: {result.height} rows x {result.width} cols")
    display(result.head(10))

flights_by_route: 592 rows x 6 cols


route_code,route_origin,route_destination,total_flights,avg_distance_km,avg_duration_min
str,str,str,u32,f64,f64
"""R470""","""FCO""","""GRX""",5862,1465.0,124.0
"""R223""","""LIL""","""MAD""",5862,1236.0,108.0
"""R075""","""BCN""","""LIL""",5862,1034.0,93.0
"""R207""","""LGW""","""NAP""",5862,1593.0,133.0
"""R156""","""CDG""","""GRX""",5862,1411.0,120.0
"""R160""","""LIL""","""GRX""",5862,1585.0,133.0
"""R313""","""JFK""","""TPA""",5862,1621.0,135.0
"""R399""","""BLQ""","""LHR""",5862,1164.0,103.0
"""R214""","""NAP""","""LHR""",5862,1631.0,136.0


flights_by_continent: 3 rows x 3 cols


origin_continent,total_flights,total_distance_km
str,u32,i64
"""EUROPE""",1066884,3351451830
"""ASIA""",351746,2331976292
"""AMERICA""",340008,2326906164


fleet_usage: 272 rows x 4 cols


airplane,airplane_model,total_flights,total_distance_km
str,str,u32,i64
"""IE19325""","""BOMBARDIER CRJ-900""",23448,6166824
"""IE28794""","""BOMBARDIER CRJ-900""",23448,6975780
"""IE30461""","""BOMBARDIER CRJ-900""",23448,6975780
"""IE22713""","""BOMBARDIER CRJ-900""",23448,6166824
"""IE38071""","""BOMBARDIER CRJ-900""",17586,8957136
"""IE34630""","""BOMBARDIER CRJ-200""",17586,8957136
"""IE89667""","""BOMBARDIER CRJ-900""",17586,20511138
"""IE85242""","""BOMBARDIER CRJ-200""",17586,20511138
"""IE02810""","""AIRBUS A330-200(332)""",15073,31596742


fleet_maintenance: 272 rows x 6 cols


airplane,airplane_model,airplane_build_date,airplane_maintenance_flight_hours,airplane_total_flight_distance,aircraft_age_years
str,str,date,i64,i64,i32
"""IE24037""","""MCDONNEL DOUGLAS DC-10-30""",2004-09-03,618,714650,22
"""IE07829""","""BOMBARDIER CRJ-900""",2001-03-30,932,946269,25
"""IE33573""","""AIRBUS A330-200(332)""",2006-08-09,2807,786988,20
"""IE02133""","""AIRBUS A321(321)""",2000-06-13,630,917907,26
"""IE00743""","""AIRBUS A340-600(346) L1""",2000-02-15,938,169615,26
"""IE12623""","""AIRBUS A340-600(346) L1""",2001-12-29,2323,126038,25
"""IE53994""","""AIRBUS A321-200(321)""",2010-11-25,1834,909017,16
"""IE85013""","""BOMBARDIER CRJ-200""",2018-08-09,2515,733801,8
"""IE09290""","""AIRBUS A330-300(333) L1""",2001-06-10,1581,540755,25
